# Accounts Receivable (AR) Synthetic Data Pipeline
This notebook demonstrates a high-performance data engineering approach to generating large-scale synthetic datasets (950K+ records) using vectorized operations with NumPy and Pandas, structured for a typical Star Schema (Fact and Dimension tables).

In [5]:
!pip install pandas numpy faker

## 1. Environment Setup
Importing core libraries for data manipulation, synthetic record generation, and performance tracking.

In [6]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import time

In [7]:
# قياس وقت التنفيذ
start_time = time.time()

fake = Faker()

In [8]:
# 1. إعداد المتغيرات وحجم البيانات
NUM_CUSTOMERS = 10000
NUM_INVOICES = 950000

print("Generating Customers Dimension...")

Generating Customers Dimension...


In [9]:
# توليد جدول العملاء باستخدام Faker (حجم صغير نسبياً)
customers_df = pd.DataFrame({
    'CustomerID': [f'C-{(i+1):05d}' for i in range(NUM_CUSTOMERS)],
    'CustomerName': [fake.company() for _ in range(NUM_CUSTOMERS)],
    'PaymentTerms': np.random.choice([30, 60, 90, 120], NUM_CUSTOMERS)
})

print("Generating Invoices Fact Table using NumPy (High Speed)...")

Generating Invoices Fact Table using NumPy (High Speed)...


In [11]:
# توليد جدول الفواتير الضخم باستخدام NumPy للسرعة القصوى
# توليد تواريخ عشوائية خلال العامين الماضيين
end_date = datetime.today()
start_date = end_date - timedelta(days=730)
date_range_days = (end_date - start_date).days

random_days_to_add = np.random.randint(0, date_range_days, NUM_INVOICES)

In [13]:
# Vectorized date generation
posting_dates = start_date + pd.to_timedelta(random_days_to_add, unit='D')

invoices_df = pd.DataFrame({
    'InvoiceID': [f'INV-{(i+1):07d}' for i in range(NUM_INVOICES)],
    'CustomerID': np.random.choice(customers_df['CustomerID'], NUM_INVOICES),
    'PostingDate': posting_dates,
    'Amount': np.round(np.random.uniform(100.0, 150000.0, NUM_INVOICES), 2),
    'Status': np.random.choice(['Open', 'Partial', 'Cleared'], NUM_INVOICES, p=[0.4, 0.2, 0.4])
})

print("Generating Bank 2nd Set Tracking Table...")

Generating Bank 2nd Set Tracking Table...


In [14]:
# استخراج الفواتير المفتوحة أو الجزئية لتوليد متابعة البنك
pending_invoices = invoices_df[invoices_df['Status'].isin(['Open', 'Partial'])]['InvoiceID'].values
NUM_BANK_DOCS = int(len(pending_invoices) * 0.7) # افتراض 70% منها ذهبت للبنك

bank_docs_df = pd.DataFrame({
    'DocID': [f'DOC-{(i+1):06d}' for i in range(NUM_BANK_DOCS)],
    'InvoiceID': np.random.choice(pending_invoices, NUM_BANK_DOCS, replace=False),
    'BankSubmissionDate': posting_dates[:NUM_BANK_DOCS] + pd.to_timedelta(np.random.randint(5, 30, NUM_BANK_DOCS), unit='D'),
    'DocStatus': np.random.choice(['Under Review', 'Accepted', 'Rejected'], NUM_BANK_DOCS, p=[0.6, 0.3, 0.1])
})

In [15]:
# التصدير إلى ملفات CSV
print("Exporting to CSV...")
customers_df.to_csv('Customers_Master.csv', index=False)
invoices_df.to_csv('AR_Invoices_950K.csv', index=False)
bank_docs_df.to_csv('Bank_Documents_Tracking.csv', index=False)

print(f"Done! Total execution time: {round(time.time() - start_time, 2)} seconds.")

Exporting to CSV...
Done! Total execution time: 1362.75 seconds.


## 3. Data Quality Assurance (DQA)
As a best practice in data engineering, we must verify the integrity of the generated data before deployment or consumption.

In [16]:
print("--- Data Summary ---")
print(f"Invoices Count: {len(invoices_df):,}")
print(f"Customers Count: {len(customers_df):,}")
print(f"Bank Documents Count: {len(bank_docs_df):,}")

# Validate Referential Integrity
missing_customers = invoices_df[~invoices_df['CustomerID'].isin(customers_df['CustomerID'])]
print(f"Referential Integrity Issues: {len(missing_customers)}")

# Preview Data
display(invoices_df.head())
display(invoices_df.describe())


--- Data Summary ---
Invoices Count: 950,000
Customers Count: 10,000
Bank Documents Count: 399,179
Referential Integrity Issues: 0


,InvoiceID,CustomerID,PostingDate,Amount,Status
0,INV-0000001,C-05805,2024-09-21 16:12:26.676785,75026.75,Cleared
1,INV-0000002,C-06779,2024-10-18 16:12:26.676785,62206.50,Open
2,INV-0000003,C-05305,2025-12-17 16:12:26.676785,127777.59,Cleared
3,INV-0000004,C-05389,2024-05-24 16:12:26.676785,116435.39,Cleared
4,INV-0000005,C-04833,2024-07-11 16:12:26.676785,37206.68,Partial


,PostingDate,Amount
count,950000,950000.000000
mean,2025-04-20 01:44:01.019522304,75045.415453
min,2024-04-20 16:12:26.676785,100.000000
25%,2024-10-19 16:12:26.676784896,37585.122500
50%,2025-04-19 16:12:26.676784896,75006.420000
75%,2025-10-18 16:12:26.676784896,112474.090000
max,2026-04-19 16:12:26.676785,149999.740000
std,NaN,43266.987427
